# cluster04 bike_change 2차 피처 조합 비교

## 용어 설명

- `suburban residential pattern`(외곽 주거 리듬): 야간, 주말, 휴일 전날처럼 주거형 리듬을 나타내는 피처 축
- `suburban location`(입지 피처): 표고, 버스정류장 수처럼 위치 특성을 나타내는 피처 축

목표:
- `cluster04`에서 경량 피처 조합이 `bike_change`에도 유지 가능한지 확인한다.


In [1]:
from pathlib import Path

import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path('/Users/cheng80/Desktop/ddri_work')
WORK_DIR = ROOT / 'works/07_prediction_bike_change'
TRAIN_PATH = WORK_DIR / 'output/data/ddri_prediction_bike_change_train_2023_2024.csv'
TEST_PATH = WORK_DIR / 'output/data/ddri_prediction_bike_change_test_2025.csv'
OUTPUT_DIR = WORK_DIR / 'output/data'

TARGET_CLUSTER = 4
TARGET_STATION_GROUP = '외곽 주거형'
RANDOM_STATE = 42


In [2]:
FULL_FEATURES = [
    'station_id', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday',
    'temperature', 'humidity', 'precipitation', 'wind_speed',
    'lag_1h', 'lag_24h', 'lag_168h', 'rolling_mean_24h', 'rolling_std_24h',
    'rolling_mean_168h', 'rolling_std_168h', 'rolling_mean_6h', 'is_weekend', 'is_night_hour',
    'is_commute_hour', 'hour_sin', 'is_rainy', 'hour_cos', 'commute_morning_flag',
    'commute_evening_flag', 'subway_distance_m', 'distance_naturepark_m', 'restaurant_count_300m',
    'convenience_store_count_300m', 'bus_stop_count_300m', 'cafe_count_300m',
    'elevation_diff_nearest_subway_m', 'nearest_park_area_sqm'
]

SUBSET_A = [
    'station_id', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday',
    'is_night_hour', 'is_weekend'
]

SUBSET_B = SUBSET_A + [
    'temperature', 'precipitation', 'is_rainy'
]

SUBSET_C = SUBSET_A + [
    'elevation_diff_nearest_subway_m', 'bus_stop_count_300m', 'subway_distance_m'
]

SUBSET_D = [
    'station_id', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday',
    'is_night_hour', 'is_weekend', 'temperature', 'precipitation', 'is_rainy',
    'elevation_diff_nearest_subway_m', 'bus_stop_count_300m', 'subway_distance_m', 'rolling_mean_6h'
]

FEATURE_SETS = {
    'full_36': FULL_FEATURES,
    'subset_a_suburban_residential_core': SUBSET_A,
    'subset_b_suburban_residential_weather': SUBSET_B,
    'subset_c_living_pattern_suburban location': SUBSET_C,
    'subset_d_current_compact': SUBSET_D,
}

BASE_CATEGORICAL = ['station_id', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday']


In [3]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

train_group = train_raw.loc[train_raw['cluster'] == TARGET_CLUSTER].copy()
test_group = test_raw.loc[test_raw['cluster'] == TARGET_CLUSTER].copy()

train_2023 = train_group.loc[train_group['date'] < '2024-01-01'].copy()
valid_2024 = train_group.loc[train_group['date'] >= '2024-01-01'].copy()
test_2025 = test_group.copy()
full_train = train_group.copy()


In [4]:
def evaluate_predictions(model_name: str, split_name: str, y_true: pd.Series, y_pred) -> dict:
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    sign_accuracy = ((pd.Series(y_pred) >= 0) == (y_true.reset_index(drop=True) >= 0)).mean()
    return {
        'model': model_name,
        'split': split_name,
        'rmse': round(float(rmse), 4),
        'mae': round(float(mae), 4),
        'r2': round(float(r2), 4),
        'sign_accuracy': round(float(sign_accuracy), 4),
    }

def prepare_xy(df: pd.DataFrame, feature_cols: list[str]):
    X = df[feature_cols].copy()
    for col in [c for c in feature_cols if c in BASE_CATEGORICAL]:
        X[col] = X[col].astype('category')
    y = df['bike_change'].copy()
    return X, y

def build_lgbm_model():
    return LGBMRegressor(
        objective='regression',
        n_estimators=400,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
    )


In [5]:
results = []

for feature_set_name, feature_cols in FEATURE_SETS.items():
    X_train, y_train = prepare_xy(train_2023, feature_cols)
    X_valid, y_valid = prepare_xy(valid_2024, feature_cols)
    X_full, y_full = prepare_xy(full_train, feature_cols)
    X_test, y_test = prepare_xy(test_2025, feature_cols)

    cat_cols = [c for c in feature_cols if c in BASE_CATEGORICAL]
    model = build_lgbm_model()
    model.fit(X_train, y_train, categorical_feature=cat_cols)
    results.append(evaluate_predictions(feature_set_name, 'train_2023', y_train, model.predict(X_train)))
    results.append(evaluate_predictions(feature_set_name, 'validation_2024', y_valid, model.predict(X_valid)))

    model = build_lgbm_model()
    model.fit(X_full, y_full, categorical_feature=cat_cols)
    results.append(evaluate_predictions(feature_set_name, 'train_full_refit', y_full, model.predict(X_full)))
    results.append(evaluate_predictions(feature_set_name, 'test_2025_refit', y_test, model.predict(X_test)))

results_df = pd.DataFrame(results)
results_df['cluster'] = TARGET_CLUSTER
results_df['station_group'] = TARGET_STATION_GROUP
results_df['feature_count'] = results_df['model'].map({k: len(v) for k, v in FEATURE_SETS.items()})

metrics_path = OUTPUT_DIR / 'ddri_bike_change_cluster04_second_round_metrics.csv'
results_df.to_csv(metrics_path, index=False, encoding='utf-8-sig')
print('saved:', metrics_path)
results_df


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.114166 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1680
[LightGBM] [Info] Number of data points in the train set: 25776, number of used features: 34


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009460 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1700
[LightGBM] [Info] Number of data points in the train set: 52128, number of used features: 34


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009246 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 60
[LightGBM] [Info] Number of data points in the train set: 25776, number of used features: 8


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003363 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 60
[LightGBM] [Info] Number of data points in the train set: 52128, number of used features: 8


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001906 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 407
[LightGBM] [Info] Number of data points in the train set: 25776, number of used features: 11


...

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002424 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 72
[LightGBM] [Info] Number of data points in the train set: 52128, number of used features: 11


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005001 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 443
[LightGBM] [Info] Number of data points in the train set: 25776, number of used features: 15


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000958 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 463
[LightGBM] [Info] Number of data points in the train set: 52128, number of used features: 15


saved: /Users/cheng80/Desktop/ddri_work/works/07_prediction_bike_change/output/data/ddri_bike_change_cluster04_second_round_metrics.csv


,model,split,rmse,mae,r2,sign_accuracy,cluster,station_group,feature_count
0,full_36,train_2023,0.6284,0.3998,0.6997,0.9065,4,외곽 주거형,35
1,full_36,validation_2024,0.7887,0.4877,0.5112,0.8901,4,외곽 주거형,35
2,full_36,train_full_refit,0.6765,0.4254,0.6462,0.9053,4,외곽 주거형,35
3,full_36,test_2025_refit,0.8638,0.5986,0.5502,0.8872,4,외곽 주거형,35
4,subset_a_suburban_residential_core,train_2023,1.0421,0.6157,0.1743,0.5556,4,외곽 주거형,9
5,subset_a_suburban_residential_core,validation_2024,1.1017,0.6446,0.0462,0.5171,4,외곽 주거형,9
6,subset_a_suburban_residential_core,train_full_refit,1.0531,0.6154,0.1427,0.5410,4,외곽 주거형,9
7,subset_a_suburban_residential_core,test_2025_refit,1.2312,0.8034,0.0862,0.5134,4,외곽 주거형,9
8,subset_b_suburban_residential_weather,train_2023,0.9999,0.5914,0.2398,0.5651,4,외곽 주거형,12
9,subset_b_suburban_residential_weather,validation_2024,1.1027,0.6452,0.0445,0.5087,4,외곽 주거형,12


In [6]:
results_df[results_df['split'] == 'test_2025_refit'].sort_values('rmse').reset_index(drop=True)


,model,split,rmse,mae,r2,sign_accuracy,cluster,station_group,feature_count
0,full_36,test_2025_refit,0.8638,0.5986,0.5502,0.8872,4,외곽 주거형,35
1,subset_d_current_compact,test_2025_refit,1.1644,0.7757,0.1826,0.6641,4,외곽 주거형,16
2,subset_b_suburban_residential_weather,test_2025_refit,1.2272,0.7966,0.0921,0.5086,4,외곽 주거형,12
3,subset_a_suburban_residential_core,test_2025_refit,1.2312,0.8034,0.0862,0.5134,4,외곽 주거형,9
4,subset_c_living_pattern_suburban location,test_2025_refit,1.2328,0.8046,0.0838,0.5190,4,외곽 주거형,12
